# ALTA 2026 — Llama-3.2-3B QLoRA fine-tuning (Kaggle T4)
Trains `sentiment` then `sarcasm` in one session. 4-bit QLoRA, no data augmentation, class-weighted focal loss. Checkpoint selection targets `min(f1_en-AU, f1_en-UK)` per task, matching the official scoring formula: `score = (min(sent-AU, sent-UK) + min(sarc-AU, sarc-UK)) / 2`.

In [1]:
!pip install -q -U "transformers==4.57.3" "peft==0.13.2" "accelerate>=0.34,<2.0" "datasets" "scikit-learn" "sentencepiece"
!pip install -q -U --no-deps bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.8 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face Hub with HF_TOKEN secret.")
except Exception as e:
    print(f"No HF_TOKEN secret found or login failed ({e}); continuing with anonymous/public access.")

Logged in to Hugging Face Hub with HF_TOKEN secret.


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import gc
import json
import datetime
import random
import zipfile
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed = 42
set_seed(seed)

root_dir = "/kaggle/working"


def _autodetect_data_dir(search_root="/kaggle/input"):
    candidates = []
    for dirpath, dirnames, filenames in os.walk(search_root):
        if "train.csv" in filenames and "valid.csv" in filenames:
            candidates.append(dirpath)
    if not candidates:
        print(f"Could not find train.csv + valid.csv anywhere under {search_root}.")
        for entry in os.listdir(search_root):
            print(" ", entry)
        raise FileNotFoundError(
            f"No folder under {search_root} contains both train.csv and valid.csv."
        )
    if len(candidates) > 1:
        print("Warning: multiple folders under /kaggle/input contain train.csv + valid.csv, using the first:")
        for c in candidates:
            print(" ", c)
    return candidates[0]


data_dir = _autodetect_data_dir()
print(f"Detected data_dir: {data_dir}")

test_csv = os.path.join(data_dir, "answer.csv")
if not os.path.exists(test_csv):
    print(f"Warning: {test_csv} not found, falling back to valid.csv as the test split.")
    test_csv = os.path.join(data_dir, "valid.csv")

model_name = "meta-llama/Llama-3.2-3B"
max_length = 128
DIALECTS = ["en-AU", "en-UK"]

task_configs = {
    "sentiment": {
        "epochs": 6,
        "batch_size": 4,
        "grad_accum": 8,
        "lr": 1e-4,
        "warmup_ratio": 0.06,
        "gamma": 0.0,
        "label_smoothing": 0.05,
        "patience": 2,
    },
    "sarcasm": {
        "epochs": 6,
        "batch_size": 4,
        "grad_accum": 8,
        "lr": 1e-4,
        "warmup_ratio": 0.06,
        "gamma": 1.0,
        "label_smoothing": 0.05,
        "patience": 3,
    },
}

run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "_llama3.2-3b_qlora"
run_dir = os.path.join(root_dir, "runs", run_id)
work_dir = os.path.join(run_dir, "tf_runs")
model_out_dir = os.path.join(run_dir, "final_models")
metrics_dir = os.path.join(run_dir, "metrics")
out = os.path.join(run_dir, "answer.csv")
submission_zip = os.path.join(run_dir, "submission.zip")

for d in (run_dir, work_dir, model_out_dir, metrics_dir):
    os.makedirs(d, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Run ID: {run_id}")
print(f"Outputs will be written under: {run_dir}")
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

run_config = {
    "run_id": run_id,
    "timestamp": datetime.datetime.now().isoformat(),
    "model_name": model_name,
    "max_length": max_length,
    "seed": seed,
    "task_configs": task_configs,
    "data_dir": data_dir,
}
with open(os.path.join(run_dir, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

Detected data_dir: /kaggle/input/datasets/asdfqwerty777/dataset
Run ID: 20260928-060222_llama3.2-3b_qlora
Outputs will be written under: /kaggle/working/runs/20260928-060222_llama3.2-3b_qlora
Using device: cuda
GPU: Tesla T4


In [4]:
class TextDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item


class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=0.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, labels):
        weight = self.alpha.to(logits.device) if self.alpha is not None else None
        ce = torch.nn.functional.cross_entropy(
            logits, labels, weight=weight, reduction="none", label_smoothing=self.label_smoothing
        )
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()


def make_trainer_class(loss_fn):
    class CustomTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            logits = outputs.logits
            loss = loss_fn(logits, labels)
            return (loss, outputs) if return_outputs else loss

    return CustomTrainer


def make_compute_metrics(dialect_array):
    dialect_array = np.asarray(dialect_array)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        labels = np.asarray(labels)

        result = {
            "macro_f1": f1_score(labels, preds, average="macro"),
            "accuracy": accuracy_score(labels, preds),
        }

        dialect_f1 = {}
        for dialect in DIALECTS:
            mask = dialect_array == dialect
            if mask.sum() > 0:
                dialect_f1[dialect] = f1_score(labels[mask], preds[mask], average="macro")
            else:
                dialect_f1[dialect] = 0.0

        result["f1_au"] = dialect_f1["en-AU"]
        result["f1_uk"] = dialect_f1["en-UK"]
        result["min_dialect_f1"] = min(dialect_f1["en-AU"], dialect_f1["en-UK"])

        return result

    return compute_metrics

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [6]:
def build_model(num_labels=2):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=torch.float16,
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        modules_to_save=["score"],
    )
    model = get_peft_model(model, lora_config)
    return model


def train_one_task(task, cfg, train_df, valid_df, test_df):
    set_seed(seed)

    train_enc = tokenizer(list(train_df["text"]), truncation=True, max_length=max_length)
    valid_enc = tokenizer(list(valid_df["text"]), truncation=True, max_length=max_length)
    test_enc = tokenizer(list(test_df["text"]), truncation=True, max_length=max_length)

    train_labels = train_df[task].tolist()

    train_ds = TextDataset(train_enc, train_labels)
    valid_ds = TextDataset(valid_enc, valid_df[task].tolist())
    test_ds = TextDataset(test_enc)

    valid_dialects = valid_df["variety"].values

    class_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_labels)
    alpha = torch.tensor(class_weights, dtype=torch.float32)

    model = build_model()
    collator = DataCollatorWithPadding(tokenizer)

    training_args = TrainingArguments(
        output_dir=os.path.join(work_dir, task),
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"] * 2,
        gradient_accumulation_steps=cfg["grad_accum"],
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=cfg["lr"],
        weight_decay=0.01,
        warmup_ratio=cfg["warmup_ratio"],
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="min_dialect_f1",
        greater_is_better=True,
        logging_steps=25,
        fp16=True,
        optim="paged_adamw_8bit",
        dataloader_num_workers=2,
        seed=seed,
        report_to=[],
        label_names=["labels"],
    )

    loss_fn = FocalLoss(alpha=alpha, gamma=cfg["gamma"], label_smoothing=cfg["label_smoothing"])
    CustomTrainer = make_trainer_class(loss_fn)
    compute_metrics = make_compute_metrics(valid_dialects)

    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["patience"])],
    )

    trainer.train()

    save_path = os.path.join(model_out_dir, task)
    os.makedirs(save_path, exist_ok=True)
    trainer.model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    with open(os.path.join(metrics_dir, f"{task}_trainer_log_history.json"), "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)

    valid_out = trainer.predict(valid_ds)
    test_out = trainer.predict(test_ds)

    valid_probs = torch.softmax(torch.tensor(valid_out.predictions), dim=-1).numpy()
    test_probs = torch.softmax(torch.tensor(test_out.predictions), dim=-1).numpy()

    valid_pred = np.argmax(valid_probs, axis=-1)
    print(f"[{task}] final validation scores (best checkpoint by min_dialect_f1):")
    overall_f1 = f1_score(valid_df[task], valid_pred, average="macro")
    print(f"  overall (pooled) macro F1: {overall_f1:.4f}")
    dialect_f1_final = {}
    for dialect in DIALECTS:
        mask = (valid_df["variety"] == dialect).values
        f1_d = f1_score(valid_df[task][mask], valid_pred[mask], average="macro")
        dialect_f1_final[dialect] = f1_d
        print(f"  {dialect}: macro F1 = {f1_d:.4f}")
    min_f1 = min(dialect_f1_final.values())
    print(f"  min(en-AU, en-UK) = {min_f1:.4f}  <- this is what the official score uses for {task}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return valid_probs, test_probs

In [7]:
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
valid_df = pd.read_csv(os.path.join(data_dir, "valid.csv"))
test_df = pd.read_csv(test_csv)

print("Class balance (train set):")
for task in ["sentiment", "sarcasm"]:
    counts = train_df[task].value_counts().to_dict()
    total = sum(counts.values())
    ratio = {k: f"{v} ({v/total:.1%})" for k, v in sorted(counts.items())}
    print(f"  {task}: {ratio}")

valid_probs_all = {}
test_probs_all = {}

for task in ["sentiment", "sarcasm"]:
    print(f"\n{'=' * 60}\nTask: {task}\n{'=' * 60}")
    vp, tp = train_one_task(task, task_configs[task], train_df, valid_df, test_df)
    valid_probs_all[task] = vp
    test_probs_all[task] = tp

valid_preds = {}
test_preds = {}
for task in ["sentiment", "sarcasm"]:
    valid_preds[task] = np.argmax(valid_probs_all[task], axis=-1)
    test_preds[task] = np.argmax(test_probs_all[task], axis=-1)
    np.save(os.path.join(metrics_dir, f"{task}_valid_probs.npy"), valid_probs_all[task])
    np.save(os.path.join(metrics_dir, f"{task}_test_probs.npy"), test_probs_all[task])

Class balance (train set):
  sentiment: {0: '2799 (50.6%)', 1: '2732 (49.4%)'}
  sarcasm: {0: '4507 (81.5%)', 1: '1024 (18.5%)'}

Task: sentiment


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

{'loss': 6.404, 'grad_norm': 53.511207580566406, 'learning_rate': 3.3333333333333335e-05, 'epoch': 0.14461315979754158}
{'loss': 3.3852, 'grad_norm': 76.04595184326172, 'learning_rate': 7.301587301587302e-05, 'epoch': 0.28922631959508316}
{'loss': 2.7578, 'grad_norm': 109.27460479736328, 'learning_rate': 9.998338935683e-05, 'epoch': 0.43383947939262474}
{'loss': 2.3556, 'grad_norm': 64.64427185058594, 'learning_rate': 9.971761008856179e-05, 'epoch': 0.5784526391901663}
{'loss': 2.537, 'grad_norm': 40.71341323852539, 'learning_rate': 9.912939304065218e-05, 'epoch': 0.7230657989877078}
{'loss': 2.1833, 'grad_norm': 16.121200561523438, 'learning_rate': 9.826487708854693e-05, 'epoch': 0.8676789587852495}
{'eval_loss': 0.26037126779556274, 'eval_macro_f1': 0.9273265201036821, 'eval_accuracy': 0.9273447820343461, 'eval_f1_au': 0.9076249340910423, 'eval_f1_uk': 0.945434350913803, 'eval_min_dialect_f1': 0.9076249340910423, 'eval_runtime': 58.0483, 'eval_samples_per_second': 13.041, 'eval_steps

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

{'loss': 5.8279, 'grad_norm': 175.4963836669922, 'learning_rate': 3.1746031746031745e-05, 'epoch': 0.14461315979754158}
{'loss': 3.9335, 'grad_norm': 82.59879302978516, 'learning_rate': 7.142857142857143e-05, 'epoch': 0.28922631959508316}
{'loss': 2.9787, 'grad_norm': 148.77015686035156, 'learning_rate': 9.998728231127501e-05, 'epoch': 0.43383947939262474}
{'loss': 3.1851, 'grad_norm': 100.80033874511719, 'learning_rate': 9.975077432032749e-05, 'epoch': 0.5784526391901663}
{'loss': 2.4786, 'grad_norm': 68.97733306884766, 'learning_rate': 9.918823949138003e-05, 'epoch': 0.7230657989877078}
{'loss': 2.5415, 'grad_norm': 58.12086868286133, 'learning_rate': 9.830670005425013e-05, 'epoch': 0.8676789587852495}
{'eval_loss': 0.23115751147270203, 'eval_macro_f1': 0.6722649508703676, 'eval_accuracy': 0.71334214002642, 'eval_f1_au': 0.7011977507998552, 'eval_f1_uk': 0.5855898842896338, 'eval_min_dialect_f1': 0.5855898842896338, 'eval_runtime': 58.2432, 'eval_samples_per_second': 12.997, 'eval_st

In [8]:
def compute_all_metrics(gold_df, run_df):
    all_metrics = {}
    dialect_scores = {}

    for task in ["sentiment", "sarcasm"]:
        all_metrics[task] = {}
        for dialect in ["en-AU", "en-UK"]:
            gold = gold_df[gold_df["variety"] == dialect][task]
            pred = run_df[run_df["variety"] == dialect][task]

            macro_f1 = f1_score(gold, pred, average="macro")
            macro_precision = precision_score(gold, pred, average="macro", zero_division=0)
            macro_recall = recall_score(gold, pred, average="macro", zero_division=0)
            acc = accuracy_score(gold, pred)
            report = classification_report(gold, pred, output_dict=True, zero_division=0)
            cm = confusion_matrix(gold, pred, labels=[0, 1])

            all_metrics[task][dialect] = {
                "accuracy": acc,
                "macro_f1": macro_f1,
                "macro_precision": macro_precision,
                "macro_recall": macro_recall,
                "classification_report": report,
                "confusion_matrix": cm.tolist(),
            }

            dialect_scores[f"{task}-{dialect}"] = macro_f1

        overall_gold = gold_df[task]
        overall_pred = run_df[task]
        all_metrics[task]["overall"] = {
            "accuracy": accuracy_score(overall_gold, overall_pred),
            "macro_f1": f1_score(overall_gold, overall_pred, average="macro"),
            "macro_precision": precision_score(overall_gold, overall_pred, average="macro", zero_division=0),
            "macro_recall": recall_score(overall_gold, overall_pred, average="macro", zero_division=0),
            "classification_report": classification_report(overall_gold, overall_pred, output_dict=True, zero_division=0),
            "confusion_matrix": confusion_matrix(overall_gold, overall_pred, labels=[0, 1]).tolist(),
        }

    min_sentiment = min(dialect_scores["sentiment-en-AU"], dialect_scores["sentiment-en-UK"])
    min_sarcasm = min(dialect_scores["sarcasm-en-AU"], dialect_scores["sarcasm-en-UK"])
    official_final_score = (min_sentiment + min_sarcasm) / 2

    official_scores = {
        **dialect_scores,
        "min_sentiment": min_sentiment,
        "min_sarcasm": min_sarcasm,
    }

    return all_metrics, official_scores, official_final_score


valid_answer_df = valid_df[["source", "variety", "text"]].copy()
valid_answer_df["sentiment"] = valid_preds["sentiment"]
valid_answer_df["sarcasm"] = valid_preds["sarcasm"]

test_answer_df = test_df[["source", "variety", "text"]].copy()
test_answer_df["sentiment"] = test_preds["sentiment"]
test_answer_df["sarcasm"] = test_preds["sarcasm"]

all_metrics, official_scores, official_final_score = compute_all_metrics(valid_df, valid_answer_df)

final_metrics = {
    "root": {
        **all_metrics,
        "official_weak_macro_f1_per_task_dialect": official_scores,
        "official_final_score": official_final_score,
    }
}

with open(os.path.join(metrics_dir, "all_metrics.json"), "w") as f:
    json.dump(final_metrics, f, indent=2)

print("Component scores:")
for name in ["sentiment-en-AU", "sentiment-en-UK", "sarcasm-en-AU", "sarcasm-en-UK"]:
    print(f"  {name}: {official_scores[name]:.4f}")
print("  min(sentiment): {:.4f}".format(official_scores["min_sentiment"]))
print("  min(sarcasm): {:.4f}".format(official_scores["min_sarcasm"]))
print(f"\nOfficial final score (valid, using min-per-task formula): {official_final_score:.4f}")

test_answer_df.to_csv(out, index=False)
with zipfile.ZipFile(submission_zip, "w") as zf:
    zf.write(out, arcname="answer.csv")

print(f"\nSaved answer.csv to: {out}")
print(f"Saved all_metrics.json to: {os.path.join(metrics_dir, 'all_metrics.json')}")
print(f"Saved submission.zip to: {submission_zip}")

Component scores:
  sentiment-en-AU: 0.9189
  sentiment-en-UK: 0.9584
  sarcasm-en-AU: 0.7845
  sarcasm-en-UK: 0.7109
  min(sentiment): 0.9189
  min(sarcasm): 0.7109

Official final score (valid, using min-per-task formula): 0.8149

Saved answer.csv to: /kaggle/working/runs/20260928-060222_llama3.2-3b_qlora/answer.csv
Saved all_metrics.json to: /kaggle/working/runs/20260928-060222_llama3.2-3b_qlora/metrics/all_metrics.json
Saved submission.zip to: /kaggle/working/runs/20260928-060222_llama3.2-3b_qlora/submission.zip
